# Brain Age Prediction from Structural MRI: Accuracy vs. Sensitivity Across Model Architectures

This notebook trains and evaluates 3D deep-learning models that predict a
person's age from a T1-weighted structural brain MRI scan ("brain age"), and
compares how well those age predictions hold up on a clinical subgroup
(participants flagged for ADHD) versus a matched group of healthy controls.

**Why this matters.** Brain-age models are widely used as a proxy for
neurological health: a predicted age that is much older than a person's
actual age (a large "brain age gap") has been associated with a range of
clinical conditions. For that proxy to be useful, the model needs to
generalize to the populations it will actually be applied to, not just to a
held-out sample that looks like the training data. This project asks a
specific architectural question — **does increasing the number of spatial
tokens available to a transformer change the model's fit and its behavior on
an unseen clinical subgroup?** — while holding the rest of the training setup
fixed.

**A note on terminology.** "Sensitivity" is used here in the clinical sense
(how the model behaves on a population it was not tuned for) rather than the
classification sense (true positive rate). This is a regression task with a
continuous age target, not a classifier, so there is no ROC-style sensitivity
metric; comparisons are made using standard regression metrics (MAE, R²,
Pearson correlation) computed separately for each group.

## 1. Dataset

The models are trained on T1-weighted structural MRI volumes from a
longitudinal pediatric/adolescent neuroimaging cohort, paired with age, sex,
and ADHD-diagnosis labels recorded at each visit. Because the same
participant can appear at multiple visits ("timepoints"), and because
participants are grouped into families with more than one member in the
cohort, a naive random train/test split risks **data leakage**: the same
brain (or a close relative's brain) appearing in both the training set and
the evaluation set.

The dataset itself is access-restricted and is **not included in this
repository**. To run this notebook you need:
- a `.mat` (HDF5 / v7.3) file containing per-row `iid` (participant id),
  `eid` (visit/timepoint id), `fid` (family id), and an `X` matrix whose rows
  are `[age, site_id, sex, ADHD1, ADHD2, ADHD3, ADHD4]`,
- a `.npy` file with the corresponding preprocessed MRI volumes, one per row.

Set the paths below to point at your own copies of these files.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import sys
sys.path.append(str(Path.cwd().parent))

from src.data import load_and_split_data
from src.engine import (train, predict, scatter_plot, set_seed,
                        save_model_with_metrics, load_model_with_metrics)
from src.model_configs import MODEL_BUILDERS, SEEDS

DATA_DIR = Path(os.environ.get("BRAIN_AGE_DATA_DIR", "../data"))
MAT_PATH = DATA_DIR / "T1_brain_age.mat"
NPY_PATH = DATA_DIR / "T1_brain_age.npy"
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Data Preparation and Leak-Free Splitting

`load_and_split_data` (in `src/data.py`) builds four groups:

1. **ADHD test set** — every visit of every participant flagged ADHD at any
   timepoint.
2. Every **family** containing an ADHD participant is excluded from every
   other split, so relatives cannot leak information across groups.
3. **Healthy test set** — age- and sex-matched controls drawn only from the
   remaining ADHD-free pool.
4. **Train / validation** — the remaining participants, split by family.

An earlier version of this pipeline matched healthy controls at a single
timepoint and removed only that one row from the training pool. Their other
visits kept the same participant id and stayed in the training set, so the
model was trained on some of the same brains it was later "tested" on. The
current version excludes by participant and by family, and prints a leakage
report confirming zero overlap between every pair of splits before training
starts.

In [ ]:
(train_dataset, val_dataset, adhd_test_dataset, healthy_test_dataset,
 train_demog, val_demog, adhd_test_demog, healthy_test_demog) = load_and_split_data(
    mat_file_path=str(MAT_PATH),
    npy_file_path=str(NPY_PATH),
    train_percentage=0.80,
    val_percentage=0.20,
    split_by="fid",
    adhd_category="ADHD4",
    healthy_ratio=3,
    age_bins=15,
)

In [ ]:
for name, demog in [("train", train_demog), ("val", val_demog),
                     ("adhd_test", adhd_test_demog), ("healthy_test", healthy_test_demog)]:
    demog.to_csv(RESULTS_DIR / f"{name}_demographics.csv", index=False)
print("Demographics saved to", RESULTS_DIR)

## 3. Model Architectures

Ten model configurations are compared, grouped into two families that share
the same input volume shape (57 x 57 x 71) and regression head design:

| Group | Models | Description |
|---|---|---|
| A-E | Hybrid CNN + Transformer | A 3D CNN feature extractor feeds a small transformer encoder. Models A-E keep the transformer and head fixed and vary how many pooling stages run before the transformer, which changes the number of spatial tokens from 2 up to 392. This is the main variable under study. A-C keep all five conv blocks (near-identical parameter counts); D and E reach 36 / 392 tokens by dropping conv blocks, so they are also much smaller models. |
| F-J | Plain 3D CNN | CNNs of varying depth and width, with no attention mechanism, included as a baseline. |

All models output a single scalar (predicted age).

In [ ]:
model_info_rows = []
for name, build in MODEL_BUILDERS.items():
    model = build()
    info = model.get_model_info()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    model_info_rows.append({
        "model": name,
        "type": info["model_type"],
        "tokens": info.get("num_tokens", "-"),
        "trainable_params": n_params,
    })
    del model

model_info_df = pd.DataFrame(model_info_rows)
print("Model configurations:")
model_info_df

## 4. Training Setup

Every model is trained from scratch with three random seeds (0, 1, 2), for
150 epochs with early stopping, using:

- **Loss:** mean squared error on age
- **Optimizer:** AdamW, learning rate 5e-6, weight decay 1e-3
- **Early stopping:** patience of 20 epochs on validation loss
- **Gradient clipping:** max norm 1.0
- **Mixed precision (AMP)** on GPU

Set `NUM_EPOCHS = 2` below for a quick smoke test of the pipeline (checks
shapes, memory, and timing, not convergence). The full setting is 150 epochs,
which is compute-intensive
(10 models x 3 seeds = 30 full training runs) and is intended to run on a
GPU.

In [ ]:
NUM_EPOCHS = 150   # set to a small number (e.g. 2) for a fast smoke test
BATCH_SIZE = 32
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 1e-3
PATIENCE = 20
CLIP_VALUE = 1.0

SMOKE_TEST = NUM_EPOCHS <= 3
SAVE_DIR = Path("../results/checkpoints_smoke" if SMOKE_TEST else "../results/checkpoints")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
if SMOKE_TEST:
    print(f"Smoke test mode: {NUM_EPOCHS} epoch(s), saving to {SAVE_DIR}")

## 5. Training

In [ ]:
results = []
total_runs = len(MODEL_BUILDERS) * len(SEEDS)
run_idx = 0

for model_name, build in MODEL_BUILDERS.items():
    for seed in SEEDS:
        run_idx += 1
        run_id = f"{model_name}_seed{seed}"
        print(f"\n{'=' * 60}\nRun {run_idx}/{total_runs}: {run_id}\n{'=' * 60}")

        set_seed(seed)
        model = build()

        model, holdout_loss, train_losses, val_losses = train(
            model, train_dataset, val_dataset, healthy_test_dataset,
            num_epochs=NUM_EPOCHS, device=device, lr=LEARNING_RATE,
            criterion_type="MSE", optimizer_type="AdamW", weight_decay=WEIGHT_DECAY,
            batch_size=BATCH_SIZE, seed=seed, patience=PATIENCE, clip_value=CLIP_VALUE,
        )

        best_val = min(val_losses)
        best_epoch = int(np.argmin(val_losses)) + 1

        save_model_with_metrics(
            model, holdout_loss, train_losses, val_losses,
            save_dir=str(SAVE_DIR), model_name=run_id,
            additional_info={"model_name": model_name, "seed": seed,
                              "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
                              "learning_rate": LEARNING_RATE},
        )

        results.append({"run_id": run_id, "model": model_name, "seed": seed,
                         "best_val_loss": best_val, "best_epoch": best_epoch,
                         "holdout_loss": float(holdout_loss)})

        del model
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_DIR / "training_summary.csv", index=False)
print("\nTraining complete for all models and seeds.")

## 6. Training Results

Average validation loss across the three seeds per model, used to compare
architectures before looking at the held-out test groups.

In [ ]:
print("Mean validation loss by model (across seeds):")
results_df.groupby("model")[["best_val_loss", "holdout_loss"]].agg(["mean", "std"]).round(4)

## 7. Evaluation on Healthy and ADHD Test Sets

This is the core comparison: every trained model (10 architectures x 3 seeds
= 30 checkpoints) is evaluated separately on the age- and sex-matched
**healthy test set** and on the **ADHD test set**, neither of which was seen
during training or used for model selection. Comparing metrics between the
two groups shows whether a model's accuracy generalizes evenly or degrades
on the clinical subgroup.

In [ ]:
healthy_metrics, adhd_metrics = {}, {}
healthy_predictions, adhd_predictions = {}, {}
healthy_labels = adhd_labels = None

for model_name in MODEL_BUILDERS:
    for seed in SEEDS:
        run_id = f"{model_name}_seed{seed}"
        checkpoint_path = SAVE_DIR / f"{run_id}.pth"
        if not checkpoint_path.exists():
            continue

        model = MODEL_BUILDERS[model_name]()
        model, _ = load_model_with_metrics(str(checkpoint_path), model)
        model = model.to(device)

        healthy_preds, healthy_y, h_metrics = predict(model, healthy_test_dataset, device=device)
        adhd_preds, adhd_y, a_metrics = predict(model, adhd_test_dataset, device=device)

        healthy_metrics[run_id] = h_metrics
        adhd_metrics[run_id] = a_metrics
        healthy_predictions[run_id] = healthy_preds
        adhd_predictions[run_id] = adhd_preds
        if healthy_labels is None:
            healthy_labels = healthy_y
        if adhd_labels is None:
            adhd_labels = adhd_y

        del model
        torch.cuda.empty_cache()

print(f"Evaluated {len(healthy_metrics)} checkpoints on both test sets.")

In [ ]:
metrics_df = pd.concat(
    [pd.DataFrame(healthy_metrics).T.assign(group="Healthy"),
     pd.DataFrame(adhd_metrics).T.assign(group="ADHD")]
)
metrics_df.index.name = "run_id"
metrics_df.to_csv(RESULTS_DIR / "per_run_metrics.csv")
metrics_df

## 8. Accuracy vs. Sensitivity Summary

Averaging each metric across the three random seeds gives one accuracy/
sensitivity comparison per architecture; the standard deviation across seeds
shows how much of any between-model difference could just be run-to-run
noise. A large gap between the healthy and
ADHD MAE for a given model indicates that its age predictions are less
reliable for that clinical subgroup — regardless of how accurate the model
looks on the healthy holdout alone.

In [ ]:
summary = (
    metrics_df.reset_index()
    .assign(model=lambda d: d["run_id"].str.replace(r"_seed\d+$", "", regex=True))
    .groupby(["model", "group"])[["MAE", "R_squared", "Pearson"]]
    .agg(["mean", "std"])
    .round(3)
)
summary

In [ ]:
pivot_mae = summary[("MAE", "mean")].unstack("group")
pivot_mae["gap_ADHD_minus_Healthy"] = pivot_mae["ADHD"] - pivot_mae["Healthy"]
pivot_mae.sort_values("gap_ADHD_minus_Healthy")

## 9. Predicted vs. Actual Age

A scatter plot for the model with the lowest mean validation loss, on both
test groups, to visually check for systematic bias (e.g. the model
compressing predictions toward the mean age, a common pattern in brain-age
regression known as regression-to-the-mean).

In [ ]:
best_model_name = results_df.groupby("model")["best_val_loss"].mean().idxmin()
best_run_id = f"{best_model_name}_seed{SEEDS[0]}"
print(f"Lowest mean validation loss: {best_model_name}")

scatter_plot(healthy_predictions[best_run_id], healthy_labels,
             title=f"{best_run_id} - Healthy test set")
scatter_plot(adhd_predictions[best_run_id], adhd_labels,
             title=f"{best_run_id} - ADHD test set")

## 10. Merged Predictions Table

Combines demographics with every model's predictions for both test groups,
useful for downstream analysis (e.g. brain-age-gap vs. clinical variables).

In [ ]:
test_demog = pd.concat(
    [healthy_test_demog.assign(group="Healthy"), adhd_test_demog.assign(group="ADHD")],
    ignore_index=True,
)

healthy_pred_df = pd.DataFrame(healthy_predictions)
adhd_pred_df = pd.DataFrame(adhd_predictions)
healthy_pred_df["age"] = healthy_labels
adhd_pred_df["age"] = adhd_labels

predictions_df = pd.concat([healthy_pred_df.assign(group="Healthy"),
                             adhd_pred_df.assign(group="ADHD")], ignore_index=True)

merged = pd.concat([test_demog.reset_index(drop=True), predictions_df.drop(columns=["group"])], axis=1)
merged.to_csv(RESULTS_DIR / "predictions_with_demographics.csv", index=False)
merged.head()

## 11. Interpretation

The cells above compute, rather than assert, the comparison this project set
out to make. Read the token-count trend in the `pivot_mae` table (Section 8)
together with the parameter counts in Section 3: models A-C share the same
transformer, head, and conv depth and have nearly identical parameter counts,
so A -> B -> C is the cleanest token-count comparison. D and E reach the same
token counts as B and C with fewer conv blocks (and far fewer parameters), so
B vs. D and C vs. E separate token count from CNN depth. A monotonic change in
the accuracy/sensitivity gap across A -> B -> C would support token count as
the driver; a non-monotonic or flat pattern would
suggest other factors (e.g. depth, parameter count) dominate instead. The
plain-CNN baselines (F-J) provide a no-attention reference point for the
same comparison.

Two limitations to keep in mind when reading these results:
- With only 3 seeds per architecture, standard deviations in Sections 6 and 8 should
  be treated as a rough estimate of run-to-run variance, not a significance
  test.
- The healthy test set is matched to the ADHD group by age and sex only; it
  does not control for other variables (e.g. site, socioeconomic factors)
  that could also differ between groups and affect the comparison.

## 12. Conclusions

Summarize here, after running the pipeline on the full dataset:
- Which architecture had the lowest validation loss, and whether that model
  also had the smallest healthy/ADHD accuracy gap (they are not guaranteed
  to be the same model).
- Whether increasing transformer token count improved fit, worsened it, or
  had no clear effect, based on the trend across models A-E.
- Whether the attention-based models outperformed the plain-CNN baselines
  (F-J) by enough to justify their added complexity.